# Hürlimann credible loss ratio reserving: verification against the paper

Working through Hürlimann, *Credible Loss Ratio Claims Reserves: The Benktander, Neuhaus and Mack Methods Revisited*, Astin Bulletin 39(1), 81-99 ([paper](https://www.casact.org/sites/default/files/2021-03/7_Hurlimann.pdf)), for [issue #1502](https://github.com/casact/chainladder-python/issues/1502).

The paper's Section 7 gives a full worked example (Mack 1997, Table 3.1.5.1). This notebook reproduces its Tables 7.2 to 7.5 one equation at a time, and asserts each result against the printed values. That gives a reference to build and test a `HurlimannCLR` estimator against.

Where the existing `cl.Benktander` and `cl.CapeCod` estimators overlap with the paper, that is shown too (Step 8). The estimator sketch is in Step 9.

| Step | Paper | What it does |
| --- | --- | --- |
| 1 | Table 7.1 | The paid triangle and the exposures |
| 2 | eq. 2.1 | The loss ratio pattern $\hat m_k$ |
| 3 | eq. 2.3, 2.4 | Payout and reserve factors $p_i$, $q_i$ |
| 4 | eq. 2.5 to 2.8 | Individual and collective reserves |
| 5 | eq. 3.2, 3.3, 6.1, 6.5 | Credibility weights: Benktander, Neuhaus, optimal |
| 6 | Tables 7.2 to 7.4 | Credible reserves and ultimates |
| 7 | eq. 4.11, Table 7.5 | Mean squared errors |
| 8 | Remark 6.1 | Overlap with `cl.Benktander` and `cl.CapeCod` |
| 9 | issue #1502 | Estimator sketch |

In [1]:
import chainladder as cl
import numpy as np
import pandas as pd

# The paper prints whole numbers; tolerance for comparing against its tables.
def close(actual, printed, atol):
    assert np.allclose(actual, printed, atol=atol), (np.round(actual, 5), printed)

## Step 1: The data (Table 7.1)

$S_{ik}$ is the paid claim amount from origin period $i$ reported in development period $k$, and $C_{ik}=\sum_{j\le k}S_{ij}$ is the cumulative. The paper's triangle is incremental. The exposure $V_i$ is premium and comes from Table 7.2.

The paper's periods are numbered $1\ldots 6$, so these are labelled 2001 to 2006, with annual development.

In [2]:
rows = [
    [4370, 1923, 3999, 2168, 1200, 647],
    [2701, 2590, 1871, 1783,  393],
    [4483, 2246, 3345, 1068],
    [3254, 2550, 2547],
    [8010, 4108],
    [5582],
]
long = pd.DataFrame(
    [(2001 + i, 2001 + i + k, v) for i, r in enumerate(rows) for k, v in enumerate(r)],
    columns=["origin", "valuation", "paid"],
)
incremental = cl.Triangle(
    long, origin="origin", valuation="valuation", columns="paid", cumulative=False
)
incremental

,12,24,36,48,60,72
2001,"4,370","1,923","3,999","2,168","1,200",647
2002,"2,701","2,590","1,871","1,783",393,
2003,"4,483","2,246","3,345","1,068",,
2004,"3,254","2,550","2,547",,,
2005,"8,010","4,108",,,,
2006,"5,582",,,,,


In [3]:
# Exposures V_i (Table 7.2), as a Triangle too so it can be used as sample_weight later.
V = pd.Series([13085, 14258, 16114, 15142, 16905, 20224], index=range(2001, 2007), dtype=float)
premium = cl.Triangle(
    pd.DataFrame({"origin": V.index, "valuation": 2006, "premium": V.values}),
    origin="origin", valuation="valuation", columns="premium", cumulative=True,
)

cumulative = incremental.incr_to_cum()
C = cumulative.latest_diagonal.to_frame().values.ravel()   # C_{i,n-i+1}
n = incremental.shape[2]
pd.DataFrame({"V": V, "C (latest paid)": C})

,V,C (latest paid)
2001,13085.0,14307.0
2002,14258.0,9338.0
2003,16114.0,11142.0
2004,15142.0,8351.0
2005,16905.0,12118.0
2006,20224.0,5582.0


## Step 2: The loss ratio pattern (eq. 2.1)

$$\hat m_k=\frac{\sum_{i=1}^{n-k+1}S_{ik}}{\sum_{i=1}^{n-k+1}V_i},\qquad k=1,\ldots,n$$

For each development period, this is the incremental paid claims from the origins observed at that age divided by those same origins' premium. Unlike a chain-ladder pattern, it is a pattern of **loss ratios**, so it depends on the exposure. Its sum $\sum_k\hat m_k$ is the burning cost loss ratio.

In [4]:
inc = incremental.values[0, 0]          # origin x development, NaN below the diagonal
observed = ~np.isnan(inc)

m = np.array([np.nansum(inc[:, k]) / V.values[observed[:, k]].sum() for k in range(n)])
pd.Series(m, index=incremental.development, name="m_k").round(5)

development
12    0.29667
24    0.17770
36    0.20072
48    0.11549
60    0.05826
72    0.04945
Name: m_k, dtype: float64

## Step 3: Payout and reserve factors (eq. 2.3, 2.4)

$$p_i=\frac{\sum_{k=1}^{n-i+1}\hat m_k}{\sum_{k=1}^{n}\hat m_k},\qquad q_i=1-p_i$$

$p_i$ is the proportion of origin period $i$'s ultimate that is expected to have been paid by now. The burning cost ultimate is $U_i^{BC}=V_i\sum_k\hat m_k$ (eq. 2.2, 2.7).

In [5]:
n_obs = observed.sum(axis=1)                       # n - i + 1 development periods seen
p = m.cumsum()[n_obs - 1] / m.sum()
q = 1 - p
U_bc = V.values * m.sum()

pd.DataFrame({"p": p, "q": q, "U_BC": U_bc}, index=V.index).round(5)

,p,q,U_BC
2001,1.00000,0.00000,11754.15793
2002,0.94496,0.05504,12807.85508
2003,0.88010,0.11990,14475.08604
2004,0.75153,0.24847,13601.94569
2005,0.52808,0.47192,15185.63545
2006,0.33026,0.66974,18167.06840


## Step 4: Individual and collective reserves (eq. 2.5 to 2.8)

The individual reserve grosses up the latest paid by the payout factor. This is the chain-ladder idea, but with the loss ratio lag factor:

$$U_i^{ind}=\frac{C_{i,n-i+1}}{p_i},\qquad R_i^{ind}=q_i U_i^{ind}=\frac{q_i}{p_i}C_{i,n-i+1}$$

The collective reserve ignores the origin's own paid claims and uses only the burning cost. This is the loss ratio method of Mack (1997):

$$R_i^{coll}=q_iU_i^{BC}$$

In [6]:
R_ind = q / p * C
R_coll = q * U_bc
pd.DataFrame({"R_ind": R_ind, "R_coll": R_coll}, index=V.index).round(0)

,R_ind,R_coll
2001,0.0,0.0
2002,544.0,705.0
2003,1518.0,1736.0
2004,2761.0,3380.0
2005,10829.0,7166.0
2006,11320.0,12167.0


## Step 5: Credibility weights (eq. 3.1 to 3.3, 6.1, 6.5)

The credible reserve mixes the two extremes, with $Z_i$ the weight on the individual reserve:

$$R_i^c=Z_iR_i^{ind}+(1-Z_i)R_i^{coll}$$

All three weights below can be written as $Z_i=p_i/(p_i+t_i)$, which is how the paper compares them:

| Method | $Z_i$ | $t_i$ |
| --- | --- | --- |
| Benktander (3.2) | $p_i$ | $q_i$ |
| Neuhaus (3.3) | $\sum_{k\le n-i+1}\hat m_k = p_i\sum_k\hat m_k$ | $q_i+\dfrac{1-\sum_k\hat m_k}{\sum_k\hat m_k}$ |
| Optimal (6.1, 6.5) | $\dfrac{p_i}{p_i+\sqrt{p_i}}=\dfrac{\sqrt{p_i}}{1+\sqrt{p_i}}$ | $\sqrt{p_i}$ |

The optimal weight is Theorem 6.1 with $f_i=1$, which minimizes both the mean squared error and the variance of the reserve. It depends only on $p_i$, and never exceeds $1/2$ (eq. 6.4).

In [7]:
t_opt = np.sqrt(p)
Z = {
    "benktander": p,
    "neuhaus":   m.cumsum()[n_obs - 1],
    "optimal":   p / (p + t_opt),
}
pd.DataFrame(Z, index=V.index).round(5)

,benktander,neuhaus,optimal
2001,1.00000,0.89829,0.50000
2002,0.94496,0.84885,0.49292
2003,0.88010,0.79059,0.48404
2004,0.75153,0.67509,0.46435
2005,0.52808,0.47437,0.42086
2006,0.33026,0.29667,0.36495


## Step 6: Credible reserves and ultimates (Tables 7.2 to 7.4)

**Table 7.2**, the parameters. $t$ is the optimal method's $t_i=\sqrt{p_i}$:

In [8]:
# The paper's "m" column lists m_k by development period k, in the same rows as the origins.
table_7_2 = pd.DataFrame(
    {"V": V.values, "m": m, "p": p, "q": q, "t": t_opt, "Z*": Z["optimal"]},
    index=range(1, n + 1),
)
table_7_2.round(5)

,V,m,p,q,t,Z*
1,13085.0,0.29667,1.00000,0.00000,1.00000,0.50000
2,14258.0,0.17770,0.94496,0.05504,0.97209,0.49292
3,16114.0,0.20072,0.88010,0.11990,0.93814,0.48404
4,15142.0,0.11549,0.75153,0.24847,0.86691,0.46435
5,16905.0,0.05826,0.52808,0.47192,0.72669,0.42086
6,20224.0,0.04945,0.33026,0.66974,0.57469,0.36495


In [9]:
# Printed values, Table 7.2.
close(m,          [0.29667, 0.17770, 0.20072, 0.11549, 0.05826, 0.04945], 5e-6)
close(p,          [1, 0.94496, 0.88010, 0.75153, 0.52808, 0.33026], 5e-6)
close(q,          [0, 0.05504, 0.11990, 0.24847, 0.47192, 0.66974], 5e-6)
close(t_opt,      [1, 0.97209, 0.93814, 0.86691, 0.72669, 0.57469], 5e-6)
close(Z["optimal"], [0.5, 0.49292, 0.48404, 0.46435, 0.42086, 0.36495], 5e-6)

**Tables 7.3 and 7.4**, reserves and ultimates by method. The "all periods" row is the total over origins 2 to $n$, since origin 1 is fully paid out ($q_1=0$).

In [10]:
reserves = pd.DataFrame(
    {
        "collective": R_coll,
        "individual": R_ind,
        **{name: z * R_ind + (1 - z) * R_coll for name, z in Z.items()},
    },
    index=range(1, n + 1),
)
reserves = reserves[["collective", "individual", "neuhaus", "benktander", "optimal"]]
ultimates = reserves.add(C, axis=0)

table_7_3 = pd.concat([reserves.iloc[1:].sum().to_frame("all periods").T, reserves.iloc[1:]])
table_7_4 = pd.concat([ultimates.sum().to_frame("all periods").T, ultimates])
table_7_3.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,25154.0,26972.0,25913.0,25999.0,25914.0
2,705.0,544.0,568.0,553.0,626.0
3,1736.0,1518.0,1564.0,1544.0,1630.0
4,3380.0,2761.0,2962.0,2915.0,3092.0
5,7166.0,10829.0,8904.0,9101.0,8708.0
6,12167.0,11320.0,11916.0,11887.0,11858.0


In [11]:
table_7_4.round(0)

,collective,individual,neuhaus,benktander,optimal
all periods,85992.0,87810.0,86751.0,86837.0,86752.0
1,14307.0,14307.0,14307.0,14307.0,14307.0
2,10043.0,9882.0,9906.0,9891.0,9964.0
3,12878.0,12660.0,12706.0,12686.0,12772.0
4,11731.0,11112.0,11313.0,11266.0,11443.0
5,19284.0,22947.0,21022.0,21219.0,20826.0
6,17749.0,16902.0,17498.0,17469.0,17440.0


In [12]:
# Printed values, Table 7.3 (all periods, then origins 2 to 6) and Table 7.4.
printed_7_3 = {
    "collective": [25154, 705, 1736, 3380, 7166, 12167],
    "individual": [26972, 544, 1518, 2761, 10829, 11320],
    "neuhaus":    [25913, 568, 1564, 2962, 8904, 11916],
    "benktander": [25999, 553, 1544, 2915, 9101, 11887],
    "optimal":    [25914, 626, 1630, 3092, 8708, 11858],
}
printed_7_4 = {
    "collective": [86752, 14307, 9964, 12772, 11443, 20826, 17440],
    "individual": [87810, 14307, 9882, 12660, 11112, 22947, 16902],
    "neuhaus":    [86751, 14307, 9906, 12706, 11313, 21022, 17498],
    "benktander": [86837, 14307, 9891, 12686, 11266, 21219, 17469],
    "optimal":    [86486, 14307, 9966, 12779, 11484, 20364, 17586],
}
for method in printed_7_3:
    close(table_7_3[method].values, printed_7_3[method], 1)

# Table 7.4: individual, Neuhaus and Benktander reproduce under their own headings.
for method in ("individual", "neuhaus", "benktander"):
    close(table_7_4[method].values, printed_7_4[method], 1)

# The printed "collective" column is our *optimal* column. See the note below.
close(table_7_4["optimal"].values, printed_7_4["collective"], 1)

Tables 7.2 and 7.3 reproduce in full, to the paper's rounding. Table 7.4 reproduces for the individual, Neuhaus and Benktander columns, but **its collective and optimal columns do not**:

- The printed "collective" column (86,752; 9,964; 12,772; 11,443; 20,826; 17,440) is what the **optimal** method gives here. For example, origin 2 is $C+R^c=9{,}338+626=9{,}964$.
- The collective ultimate from eq. 2.8, $U_i^{coll}=R_i^{coll}+C_i$, is different (origin 2: $9{,}338+705=10{,}043$; all periods 85,992).
- The printed "optimal" column (86,486; 9,966; 12,779; 11,484; 20,364; 17,586) matches neither, and I could not reproduce it from any weight tried. It sums to its own printed total, so it is internally consistent, but it is inconsistent with the optimal reserves in Table 7.3: the origin 5 optimal ultimate, 20,364, implies a reserve of 8,246 against 8,708 in Table 7.3.

So Table 7.4 looks mislabelled in the "collective" column, and the "optimal" column is unexplained. The reserves in Table 7.3 are the verified reference for the estimator, and Table 7.4 is checked only where it agrees.

## Step 7: Mean squared errors (eq. 4.11, Table 7.5)

Under the model in (4.4), Theorem 4.3 gives the mean squared error of each reserve up to the common factor $E[a_i^2(U_i)]$:

$$\frac{mse(R_i^{coll})}{E[a_i^2]}=q_i\Big(1+\frac{q_i}{t_i}\Big),\qquad
\frac{mse(R_i^{ind})}{E[a_i^2]}=\frac{q_i}{p_i},\qquad
\frac{mse(R_i^{c})}{E[a_i^2]}=\Big(\frac{Z_i^2}{p_i}+\frac{1}{q_i}+\frac{(1-Z_i)^2}{t_i}\Big)q_i^2$$

The factor cancels in a ratio, which is what Table 7.5 reports: each method's mse relative to the optimal one, using $f_i=1$ and $t_i=\sqrt{p_i}$.

In [13]:
def mse_factor(z):
    # (4.11), third line; the first two are its z=0 and z=1 cases.
    with np.errstate(divide="ignore", invalid="ignore"):
        return q**2 * (z**2 / p + 1 / q + (1 - z) ** 2 / t_opt)

mse = pd.DataFrame(
    {
        "collective": mse_factor(np.zeros(n)),
        "individual": mse_factor(np.ones(n)),
        **{name: mse_factor(z) for name, z in Z.items()},
    },
    index=range(1, n + 1),
).iloc[1:]
mse = mse[["collective", "individual", "neuhaus", "benktander", "optimal"]]
table_7_5 = mse.div(mse["optimal"], axis=0)
table_7_5.round(6)

,collective,individual,neuhaus,benktander,optimal
2,1.027133,1.028713,1.014146,1.022818,1.0
3,1.058037,1.065943,1.023277,1.038856,1.0
4,1.115379,1.153525,1.023764,1.044128,1.0
5,1.198610,1.376096,1.003211,1.012892,1.0
6,1.244422,1.740080,1.008555,1.002208,1.0


In [14]:
printed_7_5 = {
    "collective": [1.027133, 1.058036, 1.115378, 1.198612, 1.244417],
    "individual": [1.028713, 1.065943, 1.153525, 1.376096, 1.740080],
    "benktander": [1.022818, 1.038856, 1.044128, 1.012892, 1.002206],
    "optimal":    [1, 1, 1, 1, 1],
}
for method, printed in printed_7_5.items():
    close(table_7_5[method].values, printed, 1e-5)

Collective, individual, Benktander and optimal match Table 7.5.

**Neuhaus does not match, and I do not know why.** The paper prints `1.014146, 1.003002, 1.002692, 1.120972, 1.409648` for origins 2 to 6. Only the first agrees with the formula above. The comparison against the paper's own Neuhaus reserves in Table 7.3 suggests the formula's values are the sensible ones: the Neuhaus reserves for origins 5 and 6 (8,904 and 11,916) are close to the optimal ones (8,708 and 11,858), whereas a mse ratio of 1.12 to 1.41 would indicate they are not. Using $t^{WN}$ from eq. 6.3 in the mse formula, instead of $\sqrt{p}$, does not reproduce the printed values either. The Neuhaus row of Table 7.5 may be a misprint, but until the author's calculation is found this is an open question for the estimator's `mse_`.

In [15]:
table_7_5[["neuhaus"]].assign(paper=[1.014146, 1.003002, 1.002692, 1.120972, 1.409648]).round(6)

,neuhaus,paper
2,1.014146,1.014146
3,1.023277,1.003002
4,1.023764,1.002692
5,1.003211,1.120972
6,1.008555,1.409648


### The general parameters $f_i$ (eq. 6.6)

Everything above fixes $f_i=1$, so that $U_i$ and $U_i^{BC}$ are equally volatile. Theorem 6.1 gives $t_i^*$ for any $f_i\ge1$:

$$t_i^*=\frac{f_i-1+\sqrt{(f_i+1)(f_i-1+2p_i)}}{2}$$

At $f_i=1$ it reduces to $\sqrt{p_i}$, which is what was used above.

In [16]:
def t_star(f, p):
    return (f - 1 + np.sqrt((f + 1) * (f - 1 + 2 * p))) / 2

close(t_star(1.0, p), np.sqrt(p), 1e-12)
pd.DataFrame({f"f={f}": p / (p + t_star(f, p)) for f in (1.0, 1.5, 2.0)}, index=V.index).round(4)

,f=1.0,f=1.5,f=2.0
2001,0.5000,0.4000,0.3333
2002,0.4929,0.3909,0.3239
2003,0.4840,0.3796,0.3122
2004,0.4644,0.3544,0.2867
2005,0.4209,0.2993,0.2326
2006,0.3650,0.2306,0.1697


The weight $Z^*$ **falls** as $f$ rises, so $f=1$ gives the *largest* weight on the individual reserve among the allowed choices. The paper's prose before Theorem 6.1 says the opposite: that $f=1$ "yields the smallest credibility weights for the individual loss reserves". The formula was checked against the typeset page, so the prose looks like the error, but the estimator should not expose `f` until that is settled.

## Step 8: Overlap with `cl.Benktander` and `cl.CapeCod` (Remark 6.1)

Remark 6.1 says the usual methods fit the same credibility form with the loss ratio lag factor $p_i$ replaced by the chain-ladder lag factor $p_i^{CL}=1/F_{n-i+1}$. In particular, Cape Cod (6.12) is

$$R_i^{ind}=\frac{q_i^{CL}}{p_i^{CL}}C_{i,n-i+1},\qquad R_i^{coll}=q_i^{CL}\,LR\,V_i,\qquad LR=\frac{\sum_iC_{i,n-i+1}}{\sum_ip_i^{CL}V_i},\qquad Z_i=p_i^{CL}$$

Below, this is calculated by hand and compared with the existing estimators.

In [17]:
dev = cl.Development(average="volume").fit_transform(cumulative)
cdf = np.append(dev.cdf_.to_frame().values[0], 1.0)   # 12-Ult ... 72-Ult, no tail
p_cl = (1 / cdf)[::-1]                                 # origin 1 is the oldest, at 72 months
q_cl = 1 - p_cl

LR = C.sum() / (p_cl * V.values).sum()                 # Cape Cod loss ratio (6.12)
R_ind_cl = q_cl / p_cl * C
R_coll_cl = q_cl * LR * V.values
R_cc = p_cl * R_ind_cl + (1 - p_cl) * R_coll_cl        # Cape Cod, credibility form (6.12)

pd.DataFrame(
    {"p_CL": p_cl, "R_ind": R_ind_cl, "R_coll": R_coll_cl, "R (Z=p_CL)": R_cc},
    index=V.index,
).round(0)

,p_CL,R_ind,R_coll,R (Z=p_CL)
2001,1.0,0.0,0.0,0.0
2002,1.0,442.0,582.0,449.0
2003,1.0,1396.0,1620.0,1421.0
2004,1.0,2760.0,3396.0,2918.0
2005,1.0,11868.0,7553.0,9733.0
2006,0.0,11964.0,12452.0,12297.0


In [18]:
# cl.CapeCod uses the same ELR and returns the collective piece alone: it is Bornhuetter-Ferguson
# with the ELR estimated from the triangle, i.e. the Z=0 case of the form above.
cc = cl.CapeCod(trend=0, decay=1).fit(dev, sample_weight=premium)
close(cc.apriori_.to_frame().values.ravel(), LR, 1e-9)
close(cc.ibnr_.to_frame().values.ravel()[1:], R_coll_cl[1:], 1e-6)

# cl.Benktander with n_iters=2 is the Z=p_CL mixture (eq. 3.2), with the Cape Cod ELR as the a priori.
gb = cl.Benktander(apriori=LR, n_iters=2).fit(dev, sample_weight=premium)
close(gb.ibnr_.to_frame().values.ravel()[1:], R_cc[1:], 1e-6)

So the Cape Cod and Benktander methods already in chainladder correspond to Hürlimann's *Cape Cod* and *Benktander* methods in Remark 6.1, which use the chain-ladder pattern. What is not in the package:

- **The loss ratio pattern** $\hat m_k$ (Step 2). It is estimated from premium and incremental claims, not from link ratios. Using it changes the answer, e.g. $p$ for the latest origin is 0.330 here against $p^{CL}$ = 0.318.
- **The optimal weight** $Z^*=p/(p+\sqrt p)$, and the Neuhaus weight. `cl.Benktander` has no credibility weight other than its iteration count.
- **Optimal Cape Cod** and **Optimal Bornhuetter-Ferguson** (6.13): the same weight with $p^{CL}$ in place of $p$.
- **The mean squared error** $mse(R^c)$ of Theorem 4.3, with the pragmatic estimators of Section 5.

Optimal Cape Cod needs only a weight, and can be computed with the estimators above:

In [19]:
Z_opt_cl = p_cl / (p_cl + np.sqrt(p_cl))
R_opt_cc = Z_opt_cl * R_ind_cl + (1 - Z_opt_cl) * R_coll_cl
pd.DataFrame({"Z (6.13)": Z_opt_cl, "Optimal Cape Cod": R_opt_cc}, index=V.index).round(4)

,Z (6.13),Optimal Cape Cod
2001,0.5000,0.0000
2002,0.4942,513.0738
2003,0.4852,1511.5892
2004,0.4644,3100.7663
2005,0.4155,9345.7549
2006,0.3606,12275.9783


## Step 9: Estimator sketch

The interface requested in the issue, as a prototype that lives in this notebook only:

```python
HurlimannCLR(method="optimal").fit(tri, sample_weight=premium)
```

with `method` one of `optimal`, `benktander`, `neuhaus`, `collective` and `individual`, and attributes `p_`, `elr_`, `credibility_`, `r_individual_`, `r_collective_`, `ibnr_` and `ultimate_`. It takes the cumulative triangle and a premium Triangle, as `cl.Benktander` and `cl.CapeCod` do.

In [20]:
from sklearn.base import BaseEstimator


class HurlimannCLR(BaseEstimator):
    def __init__(self, method="optimal"):
        self.method = method

    def fit(self, X, y=None, sample_weight=None):
        inc = X.cum_to_incr().values[0, 0]
        V = sample_weight.latest_diagonal.to_frame().values.ravel()
        seen = ~np.isnan(inc)
        k = seen.sum(axis=1)

        self.m_ = np.nansum(inc, axis=0) / (seen * V[:, None]).sum(axis=0)    # eq. 2.1
        self.elr_ = self.m_.sum()                                             # burning cost loss ratio
        self.p_ = self.m_.cumsum()[k - 1] / self.elr_                         # eq. 2.3
        C = np.nansum(inc, axis=1)
        q = 1 - self.p_

        self.r_individual_ = q / self.p_ * C                                  # eq. 2.6
        self.r_collective_ = q * V * self.elr_                                # eq. 2.7
        self.credibility_ = {
            "individual": np.ones_like(self.p_),
            "collective": np.zeros_like(self.p_),
            "benktander": self.p_,
            "neuhaus": self.m_.cumsum()[k - 1],
            "optimal": self.p_ / (self.p_ + np.sqrt(self.p_)),
        }[self.method]
        z = self.credibility_
        self.ibnr_ = z * self.r_individual_ + (1 - z) * self.r_collective_    # eq. 3.1
        self.ultimate_ = C + self.ibnr_
        return self


for method in printed_7_3:
    est = HurlimannCLR(method=method).fit(cumulative, sample_weight=premium)
    close(est.ibnr_[1:].sum(), printed_7_3[method][0], 1)
    close(est.ibnr_[1:], printed_7_3[method][1:], 1)
    if method in ("individual", "neuhaus", "benktander"):   # the Table 7.4 columns that reproduce
        close(est.ultimate_, printed_7_4[method][1:], 1)

pd.DataFrame(
    {"p_": est.p_, "credibility_": est.credibility_, "r_individual_": est.r_individual_,
     "r_collective_": est.r_collective_, "ibnr_": est.ibnr_, "ultimate_": est.ultimate_},
    index=V.index,
).round(4)

,p_,credibility_,r_individual_,r_collective_,ibnr_,ultimate_
2001,1.0000,0.5000,0.0000,0.0000,0.0000,14307.0000
2002,0.9450,0.4929,543.9453,705.0001,625.6124,9963.6124
2003,0.8801,0.4840,1517.9336,1735.5715,1630.2260,12772.2260
2004,0.7515,0.4644,2761.0071,3379.6837,3092.3981,11443.3981
2005,0.5281,0.4209,10829.1463,7166.3581,8707.8731,20825.8731
2006,0.3303,0.3650,11319.6176,12167.1353,11857.8313,17439.8313


The prototype reproduces Table 7.3 for all five methods, and the Table 7.4 columns that reproduce.

### Open questions for the real estimator

1. **Return types.** These attributes are numpy arrays. A real estimator would return Triangles, like the other methods, so that `ibnr_`, `ultimate_` and `r_individual_` carry the origin index and work with `.sum()`, `.to_frame()` and the like.
2. **A separate loss ratio pattern step.** The pattern $\hat m_k$ could be its own estimator, e.g. `LossRatioPattern`, fit on the triangle and premium. The credibility step would then take any pattern, which is Remark 6.1: with a chain-ladder pattern it gives the optimal Cape Cod and Bornhuetter-Ferguson methods (Step 8).
3. **Enhance or add.** Optimal weights could instead be added to `cl.Benktander` as a credibility option, since Step 8 shows it already matches the paper's chain-ladder-based forms. The loss ratio pattern is the part with no counterpart in the package.
4. **`mse_`.** Theorem 4.3 and the Section 5 estimators are not implemented here. The Neuhaus mismatch in Step 7 should be settled first.
5. **`f` and $\beta$.** Only $f_i=1$ is verified against the paper's numbers (Step 7). The general case has a contradiction between eq. 6.6 and the paper's prose that should be settled first.
6. **Tail.** The paper assumes no development beyond $n$ periods. A tail factor would need a place to go.